# Evaluating the long-private-chain defense

Four questions, answered from simulation CSVs:

1. How does the double-spend success rate change with attacker hash share, with and without the defense?
2. What does the final chain look like (orphans, who owns the blocks, longest single-miner streak)?
3. What does the defense cost in runtime and memory?
4. Does the defense wrongly block large honest miners (false positives)?

**Data.** The attack grid CSV has the columns `scenario, attacker_share, attack_success, final_chain_length, reorg_depth, max_consecutive_by_one_miner, runtime_ms, memory_mb`. Orphan rate and block ownership need chain-structure counts, so this notebook generates a *detailed* grid CSV (the same eight columns plus five structure columns). False positives need an honest-only experiment, so it has its own CSV. Both are created on first run and reused afterwards; delete `results/` to regenerate.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pow51sim.analysis.defense_plots import (
    plot_chain_structure,
    plot_defense_overhead,
    plot_false_positives,
    plot_success_vs_share,
)
from pow51sim.analysis.style import apply_style
from pow51sim.analysis.tables import load_results, overhead_table
from pow51sim.simulation.false_positives import FalsePositiveExperiment
from pow51sim.simulation.simulation_runner import SimulationRunner

apply_style()
RESULTS_DIR = Path("../results")
GRID_CSV = RESULTS_DIR / "grid_detailed.csv"
FALSE_POSITIVE_CSV = RESULTS_DIR / "false_positives.csv"
SEED = 42
ATTACK_TRIALS = 200
FALSE_POSITIVE_TRIALS = 50

## Load (or generate) the data

In [ ]:
if not GRID_CSV.exists():
    SimulationRunner(ATTACK_TRIALS, seed=SEED, detailed=True).run_to_csv(GRID_CSV)
if not FALSE_POSITIVE_CSV.exists():
    experiment = FalsePositiveExperiment(FALSE_POSITIVE_TRIALS, seed=SEED)
    experiment.run_to_csv(FALSE_POSITIVE_CSV)

results = load_results(GRID_CSV)
false_positives = pd.read_csv(FALSE_POSITIVE_CSV)
print(f"{len(results):,} attack trials, {len(false_positives):,} false-positive trials")
results.head()

## 1. Attack success rate vs attacker hash share

One panel per confirmation depth. Lines show the mean success rate over trials with a 95% confidence band. The defense is `LongPrivateChainRule(6)`: it rejects the sixth consecutive block from one miner, so it only stops attacks that need a private chain of six or more blocks.

In [ ]:
fig = plot_success_vs_share(results)
plt.show()

## 2. Final chain structure

- **Orphaned blocks**: published blocks that ended up off the final chain (withheld blocks that were never released are not counted).
- **Who owns the final chain**: the attacker's share of final-chain blocks against its hash share. The dashed line is a fair share.
- **Longest streak**: the longest run of consecutive blocks by one miner on the final chain. The dashed line is the defense threshold.

In [ ]:
fig = plot_chain_structure(results, threshold=6)
plt.show()

## 3. Runtime and memory overhead of the defense

Medians per trial, pooled over confirmation depths. Memory is peak Python allocation measured with `tracemalloc`; runtime is measured in a separate un-traced run of the same seeded trial. Trials are sub-millisecond, so treat small differences as noise.

In [ ]:
fig = plot_defense_overhead(results)
plt.show()
overhead_table(results).round(3)

## 4. False positives: large honest miners wrongly blocked

An honest-only network where one miner holds the share on the x-axis and nobody attacks. Every rejection here is a false positive. Dotted lines are the closed-form rate `share^(threshold-1)`: once a miner's streak reaches `threshold-1`, all of its further wins are rejected until another miner finds a block.

In [ ]:
fig = plot_false_positives(false_positives)
plt.show()

In [ ]:
(
    false_positives.groupby(["threshold", "miner_share"])["blocked_rate"]
    .mean()
    .unstack("miner_share")
    .round(4)
)